# Balancing Methods for LVC Dataset

This notebook evaluates class imbalance handling strategies for the LVC project using:

- SMOTE
- ADASYN
- class_weight

The goal is to reduce false negatives by improving recall for the positive class.


In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

import warnings
warnings.filterwarnings("ignore")

In [13]:
DATA_PATH = "../data/raw/leish_dataset.csv"
TARGET = "diagnosis"
SEEDS = [41, 42, 46]
TEST_SIZE = 0.30
N_SPLITS = 5
MIN_BREED_COUNT = 10

In [14]:
df = pd.read_csv(DATA_PATH)

breed_counts = df["breed_name"].value_counts(dropna=False)
rare_breeds = breed_counts[breed_counts < MIN_BREED_COUNT].index
df["breed_name"] = df["breed_name"].replace(rare_breeds, "other")

print("Shape:", df.shape)
display(df.head())

Shape: (456, 17)


,diagnosis,general_state,ectoparasites,nutritional_state,coat,nails,mucosa_color,muzzle_ear_lesion,lymph_nodes,blepharitis,conjunctivitis,alopecia,bleeding,skin_lesion,muzzle_lip_depigmentation,animal_sex,breed_name
0,negativo,bom,ausente,adequado,normal,normal,normal,presente,normal,ausente,Ausente,ausente,presente,Ausente,ausente,M,SRD
1,negativo,bom,ausente,adequado,normal,normal,normal,ausente,normal,ausente,Ausente,ausente,ausente,Ausente,ausente,M,SRD
2,negativo,bom,leve,adequado,normal,normal,normal,ausente,normal,ausente,Ausente,ausente,presente,Ausente,ausente,M,Poodle
3,negativo,bom,ausente,adequado,normal,leves_moderadas,normal,ausente,normal,ausente,Ausente,ausente,ausente,Ausente,ausente,F,other
4,negativo,bom,leve,leve_moderado,normal,leves_moderadas,normal,ausente,leves_moderadas,ausente,Ausente,ausente,ausente,Ausente,ausente,F,SRD


In [15]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", list(le.classes_))
print(pd.Series(y_encoded).value_counts())

Classes: ['negativo', 'positivo']
0    320
1    136
Name: count, dtype: int64


In [16]:
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

Categorical features: ['general_state', 'ectoparasites', 'nutritional_state', 'coat', 'nails', 'mucosa_color', 'muzzle_ear_lesion', 'lymph_nodes', 'blepharitis', 'conjunctivitis', 'alopecia', 'bleeding', 'skin_lesion', 'muzzle_lip_depigmentation', 'animal_sex', 'breed_name']
Numerical features: []


In [17]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [18]:
base_models = {
    "LR": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(),
    "RF": RandomForestClassifier(random_state=42),
    "GB": GradientBoostingClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=1000, random_state=42)
}

weighted_models = {
    "LR": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "SVM": SVC(probability=True, class_weight="balanced", random_state=42),
    "RF": RandomForestClassifier(class_weight="balanced", random_state=42)
}

In [19]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

In [20]:
from sklearn.base import clone

cv_results = []
holdout_results = []

for seed in SEEDS:
    print("=" * 90)
    print(f"SEED {seed}")
    print("=" * 90)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=TEST_SIZE,
        stratify=y_encoded,
        random_state=seed
    )

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    # --------------------------
    # Group A1: RandomOverSampler
    # --------------------------
    for model_name, model in base_models.items():
        pipe = ImbPipeline(steps=[
            ("preprocessor", preprocessor),
            ("sampler", RandomOverSampler(random_state=seed)),
            ("clf", clone(model))
        ])

        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        )

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        cv_results.append({
            "seed": seed,
            "group": "RandomOverSampler",
            "model": model_name,
            "cv_accuracy_mean": np.mean(scores["test_accuracy"]),
            "cv_precision_mean": np.mean(scores["test_precision"]),
            "cv_recall_mean": np.mean(scores["test_recall"]),
            "cv_f1_mean": np.mean(scores["test_f1"])
        })

        holdout_results.append({
            "seed": seed,
            "group": "RandomOverSampler",
            "model": model_name,
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision": precision_score(y_test, y_pred, zero_division=0),
            "test_recall": recall_score(y_test, y_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_pred, zero_division=0)
        })

        print(
            f"ROS | {model_name} | "
            f"CV Recall: {np.mean(scores['test_recall']):.4f} | "
            f"Test Recall: {recall_score(y_test, y_pred, zero_division=0):.4f}"
        )

    # --------------------------
    # Group A2: class_weight
    # --------------------------
    for model_name, model in weighted_models.items():
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("clf", clone(model))
        ])

        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        )

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        cv_results.append({
            "seed": seed,
            "group": "class_weight",
            "model": model_name,
            "cv_accuracy_mean": np.mean(scores["test_accuracy"]),
            "cv_precision_mean": np.mean(scores["test_precision"]),
            "cv_recall_mean": np.mean(scores["test_recall"]),
            "cv_f1_mean": np.mean(scores["test_f1"])
        })

        holdout_results.append({
            "seed": seed,
            "group": "class_weight",
            "model": model_name,
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision": precision_score(y_test, y_pred, zero_division=0),
            "test_recall": recall_score(y_test, y_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_pred, zero_division=0)
        })

        print(
            f"CW  | {model_name} | "
            f"CV Recall: {np.mean(scores['test_recall']):.4f} | "
            f"Test Recall: {recall_score(y_test, y_pred, zero_division=0):.4f}"
        )

SEED 41
ROS | LR | CV Recall: 0.5053 | Test Recall: 0.4634
ROS | SVM | CV Recall: 0.5053 | Test Recall: 0.3171
ROS | KNN | CV Recall: 0.3474 | Test Recall: 0.1951
ROS | RF | CV Recall: 0.3684 | Test Recall: 0.3171
ROS | GB | CV Recall: 0.4632 | Test Recall: 0.3659
ROS | MLP | CV Recall: 0.4000 | Test Recall: 0.2439
CW  | LR | CV Recall: 0.5368 | Test Recall: 0.4390
CW  | SVM | CV Recall: 0.5368 | Test Recall: 0.4146
CW  | RF | CV Recall: 0.2526 | Test Recall: 0.1951
SEED 42
ROS | LR | CV Recall: 0.4737 | Test Recall: 0.5366
ROS | SVM | CV Recall: 0.5158 | Test Recall: 0.4146
ROS | KNN | CV Recall: 0.3053 | Test Recall: 0.3171
ROS | RF | CV Recall: 0.3368 | Test Recall: 0.3171
ROS | GB | CV Recall: 0.3789 | Test Recall: 0.4146
ROS | MLP | CV Recall: 0.4000 | Test Recall: 0.2927
CW  | LR | CV Recall: 0.5158 | Test Recall: 0.5366
CW  | SVM | CV Recall: 0.5684 | Test Recall: 0.6098
CW  | RF | CV Recall: 0.1684 | Test Recall: 0.1463
SEED 46
ROS | LR | CV Recall: 0.4632 | Test Recall: 0.5854

In [21]:
cv_results_df = pd.DataFrame(cv_results)
holdout_results_df = pd.DataFrame(holdout_results)

display(cv_results_df.head())
display(holdout_results_df.head())

,seed,group,model,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean
0,41,RandomOverSampler,LR,0.651984,0.423487,0.505263,0.459136
1,41,RandomOverSampler,SVM,0.686310,0.468252,0.505263,0.484795
2,41,RandomOverSampler,KNN,0.658085,0.430823,0.347368,0.380878
3,41,RandomOverSampler,RF,0.661310,0.432177,0.368421,0.393136
4,41,RandomOverSampler,GB,0.670933,0.451417,0.463158,0.456002


,seed,group,model,test_accuracy,test_precision,test_recall,test_f1
0,41,RandomOverSampler,LR,0.591241,0.358491,0.463415,0.404255
1,41,RandomOverSampler,SVM,0.598540,0.325000,0.317073,0.320988
2,41,RandomOverSampler,KNN,0.620438,0.296296,0.195122,0.235294
3,41,RandomOverSampler,RF,0.605839,0.333333,0.317073,0.325000
4,41,RandomOverSampler,GB,0.576642,0.319149,0.365854,0.340909


In [22]:
cv_summary = (
    cv_results_df.groupby(["group", "model"])[[
        "cv_accuracy_mean", "cv_precision_mean", "cv_recall_mean", "cv_f1_mean"
    ]]
    .mean()
    .sort_values(by=["group", "cv_recall_mean"], ascending=[True, False])
)

holdout_summary = (
    holdout_results_df.groupby(["group", "model"])[[
        "test_accuracy", "test_precision", "test_recall", "test_f1"
    ]]
    .mean()
    .sort_values(by=["group", "test_recall"], ascending=[True, False])
)

print("CV Summary")
display(cv_summary)

print("Holdout Summary")
display(holdout_summary)

CV Summary


cv_accuracy_mean  cv_precision_mean  cv_recall_mean  \
group             model                                                        
RandomOverSampler SVM            0.618436           0.396524        0.484211   
                  LR             0.587335           0.360683        0.480702   
                  GB             0.587269           0.343930        0.392982   
                  MLP            0.616452           0.360272        0.361404   
                  RF             0.619626           0.352792        0.315789   
                  KNN            0.624702           0.360476        0.312281   
class_weight      SVM            0.600694           0.390570        0.540351   
                  LR             0.597751           0.376920        0.508772   
                  RF             0.648892           0.370811        0.214035   

                         cv_f1_mean  
group             model              
RandomOverSampler SVM      0.430156  
                  LR       0.409448  
                  GB       0.363727  
                  MLP      0.355721  
                  RF       0.326434  
                  KNN      0.331520  
class_weight      SVM      0.448701  
                  LR       0.431262  
                  RF       0.264324

Holdout Summary


test_accuracy  test_precision  test_recall   test_f1
group             model                                                      
RandomOverSampler LR          0.639903        0.420423     0.528455  0.467967
                  GB          0.637470        0.403856     0.430894  0.416223
                  SVM         0.637470        0.395155     0.406504  0.400647
                  RF          0.630170        0.373294     0.341463  0.356566
                  KNN         0.608273        0.329877     0.317073  0.318633
                  MLP         0.639903        0.375220     0.308943  0.338528
class_weight      SVM         0.622871        0.399807     0.536585  0.457070
                  LR          0.632603        0.409502     0.512195  0.455119
                  RF          0.661800        0.404046     0.170732  0.233206

In [23]:
best_recall_cv = cv_summary["cv_recall_mean"].idxmax()
best_recall_holdout = holdout_summary["test_recall"].idxmax()

print("Best scenario by CV recall:", best_recall_cv)
print("Best scenario by Holdout recall:", best_recall_holdout)

Best scenario by CV recall: ('class_weight', 'SVM')
Best scenario by Holdout recall: ('class_weight', 'SVM')


In [24]:
cv_results_df.to_csv("../results/tables/balancing_cv_results.csv", index=False)
holdout_results_df.to_csv("../results/tables/balancing_holdout_results.csv", index=False)
cv_summary.to_csv("../results/tables/balancing_cv_summary.csv")
holdout_summary.to_csv("../results/tables/balancing_holdout_summary.csv")

print("Balancing results saved to ../results/tables/")

Balancing results saved to ../results/tables/


## Notes

- Since all original variables are categorical, preprocessing is applied before resampling.
- SMOTENC is used instead of standard SMOTE because the original data are categorical.
- ADASYN is included for comparison after encoded transformation.
- The main evaluation focus is recall, since false negatives are especially harmful in LVC screening.
